# The scene turns with the reasoning

Sections 3 and 5.3 (Fig. 3b, Fig. 6b, Table 3): the viewpoint probe inserts the probe tokens of `03_views_fused.ipynb` after
each viewpoint that a held-out reasoning chain declares, and asks whether their states hold the scene **as seen from
that viewpoint**. It reports the balanced accuracy over the four directions around the viewer, and four controls: the
target of another viewpoint of the chain, the tokens read before the declaration, all-zero states, and shuffled targets.

**Needs:** the probe tokens of `03_views_fused.ipynb` and the perspective-stage rows with their held-out split
(`scripts/data/sft.sh`); one GPU, about an hour per model.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import smind_nb as nb
nb.setup()

SIZE = "4b"   # or "9b"
GPU = "0"

In [ ]:
nb.need("data/smind/sft/gpt-6-astra/perspective/heldout_z.parquet")
MODELS = ["base", "s-mind"]
for name in MODELS:
    d = nb.RUNS / f"{SIZE}-probe/{name}"
    nb.need(str((d / "probe/ckpt").relative_to(nb.ROOT)), build="03_views_fused.ipynb")
    nb.run(["python", "analysis/diagnosis/probe_view.py", "--ckpt", d / "probe/ckpt", "--out", d / "view_probe.json"],
           d / "view_probe.log", done=d / "view_probe.json", gpu=GPU)

In [ ]:
import pandas as pd
KEY = "quad_bal/other_image/{}"
def direction(v, q):
    rows = [r[f"quad{q}/other_image/all"] for r in v["per_row_s_k"] if f"quad{q}/other_image/all" in r]
    return 100 * sum(a for a, _ in rows) / sum(b for _, b in rows)
cols = {}
for name in MODELS:
    v = nb.read_json(nb.RUNS / f"{SIZE}-probe/{name}/view_probe.json")
    a = v["arms"]
    cols[name] = {"declared viewpoint": 100 * a["s_k"][KEY.format("all")]["acc"],
                  "  camera viewpoints": 100 * a["s_k"][KEY.format("camera")]["acc"],
                  "  imagined standpoints": 100 * a["s_k"][KEY.format("turned")]["acc"],
                  "ahead": direction(v, 0), "right": direction(v, 1), "behind": direction(v, 2), "left": direction(v, 3),
                  "control: another viewpoint's target": 100 * a["wrong_view"][KEY.format("all")]["acc"],
                  "control: before the declaration": 100 * a["s_0"][KEY.format("all")]["acc"],
                  "control: zero states": 100 * a["zero"][KEY.format("all")]["acc"],
                  "control: shuffled targets": 100 * a["shuffled"][KEY.format("all")]["acc"]}
df = pd.DataFrame(cols).rename(columns={"base": "Base", "s-mind": "S-Mind (SFT)"})
if SIZE == "4b":
    df["paper: Base"] = pd.Series({"declared viewpoint": 32.40, "  camera viewpoints": 31.76,
                                   "  imagined standpoints": 33.61, "ahead": 88.67, "right": 35.72, "behind": 3.29,
                                   "left": 1.90})
    df["paper: S-Mind (SFT)"] = pd.Series({"declared viewpoint": 82.17, "  camera viewpoints": 85.46,
                                           "  imagined standpoints": 73.90, "ahead": 96.51, "right": 85.29,
                                           "behind": 77.38, "left": 69.51,
                                           "control: another viewpoint's target": 30.10,
                                           "control: before the declaration": 29.43})
display(df)

Both models recover what lies ahead of the declared viewer, but only S-Mind recovers what lies behind and to the sides;
Base stays at the level of the controls. In S-Mind the decoded scene is bound to the declaration: it falls to the
controls when another viewpoint is the target or when the tokens are read before the viewpoint is declared.